# Prepare

In [1]:
! pip install -q -U -q transformers torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2

In [2]:
import gc
from transformers import AutoModelForCausalLM, AutoProcessor ,AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import numpy as np
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from typing import List, Optional
import kagglehub
from PIL import Image
import requests
import tempfile
import librosa

# Gemma Manager

In [3]:
class GemmaManager:
    def __init__(
        self,
        dtype=torch.bfloat16,
        max_new_tokens=2048
    ):
        try:
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception as e:
            print(e)

        self.model_path = None
        self.dtype = dtype
        self.max_new_tokens = max_new_tokens

        self._setup()

    def _setup(self):
        self.model_path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")
        self.processor = AutoProcessor.from_pretrained(self.model_path)

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            dtype=self.dtype,
            device_map="auto"
        )

        self.device = self.model.device

    def ask(self, prompt, enable_thinking=False):
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ]

        text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking
        )

        inputs = self.processor(text=text, return_tensors="pt")
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens
            )

        generated_ids = outputs[0]
        input_len = inputs["input_ids"].shape[-1]

        new_tokens = generated_ids[input_len:] if len(generated_ids) > input_len else generated_ids

        raw_output = self.processor.decode(new_tokens, skip_special_tokens=False)

        thinking = None
        response = raw_output

        try:
            parsed = self.processor.parse_response(raw_output)

            if isinstance(parsed, dict):
                response = parsed.get("content", raw_output)
                thinking = parsed.get("thinking", None)

        except Exception:
            pass

        return response, thinking
    
    def ask_with_image(
        self,
        prompt,
        image,
        enable_thinking=False
    ):
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image}, 
                    {"type": "text", "text": prompt}
                ]
            }
        ]
    
        text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking
        )
    
        inputs = self.processor(
            text=text,
            images=image,
            return_tensors="pt"
        ).to(self.model.device)
    
        input_len = inputs["input_ids"].shape[-1]
    
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens
            )
    
        generated_ids = outputs[0]
        new_tokens = generated_ids[input_len:]
    
        raw_output = self.processor.decode(
            new_tokens,
            skip_special_tokens=False
        )
    
        thinking = None
        response = raw_output
    
        try:
            parsed = self.processor.parse_response(raw_output)
            if isinstance(parsed, dict):
                response = parsed.get("content", raw_output)
                thinking = parsed.get("thinking", None)
        except Exception:
            pass
    
        return response, thinking
    
    def ask_with_audio(
        self,
        prompt,
        audio,
        enable_thinking=False
    ):
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": audio},
                    {"type": "text", "text": prompt}
                ]
            }
        ]
        
        text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking
        )

        inputs = self.processor(
            text=text,
            audio=audio,
            return_tensors="pt"
        ).to(self.model.device)
    
        input_len = inputs["input_ids"].shape[-1]
    
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens
            )
    
        generated_ids = outputs[0]
        new_tokens = generated_ids[input_len:]
    
        raw_output = self.processor.decode(
            new_tokens,
            skip_special_tokens=False
        )
    
        thinking = None
        response = raw_output
    
        try:
            parsed = self.processor.parse_response(raw_output)
            if isinstance(parsed, dict):
                response = parsed.get("content", raw_output)
                thinking = parsed.get("thinking", None)
        except Exception:
            pass
    
        return response, thinking

In [4]:
def load_model():
    return GemmaManager()

gemma = load_model()

Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

# Test

In [5]:
gemma.ask(
    "Hi"
)

('Hi! How can I help you today? 😊', None)

### Image test

![](https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png)

In [6]:
img_url = "https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"
image = Image.open(requests.get(img_url, stream=True).raw).convert("RGB")

response, _ = gemma.ask_with_image(
    "What is happening in this image?",
    image=image
)

print(response)

This image features a **brown and white cow standing on a sandy beach** with the **ocean and a blue sky** in the background.

Here's a breakdown of what's happening:

*   **Subject:** The central focus is a bovine animal (a cow) with reddish-brown fur and a prominent white face/muzzle.
*   **Setting:** The cow is standing on light-colored sand, suggesting a beach environment.
*   **Background:** In the distance, there is a clear blue ocean meeting the horizon, and some green landmasses or hills are visible, likely on the far side of the water. The sky is bright blue with some white, wispy clouds.
*   **Action/Mood:** The cow appears to be standing calmly, looking towards the camera or slightly off to the side. The scene conveys a bright, sunny, and relaxed outdoor setting.

In summary, it is a photograph of a cow posing or standing on a beach by the sea.


### Audio test

[Audio link](https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/journal1.wav)

In [7]:
def download_audio(url):
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
        return f.name

audio_url = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/journal1.wav"
audio_path = download_audio(audio_url)

print("Downloaded audio at:", audio_path)

audio_array, sr = librosa.load(audio_path, sr=16000)

response, thinking = gemma.ask_with_audio(
    "Transcribe the following audio exactly. Only output transcription.",
    audio=audio_array
)

print("\n--- RESPONSE ---")
print(response)

if thinking:
    print("\n--- THINKING ---")
    print(thinking)

Downloaded audio at: /tmp/tmphrzib0mm.wav

--- RESPONSE ---
I woke up today feeling really fresh. The morning light was beautiful and I enjoyed a nice cup of coffee.
